# 🎙️ Whisper-Medium Fine-Tune — Kannada + Hindi + English (v2)
**Target:** WER < 15% (Accuracy > 85%) for kn / hi / en

**Before running:**
1. Runtime → Change runtime type → **A100 GPU** (uses ~30 compute units)
2. Make sure you are signed into the Google account that has your Drive datasets
3. Run cells **in order**, top to bottom
4. Training takes **~3–4 hours** on A100

**Model published to:** `udayakumar8214/whisper-classroom-kn-hi-en-v2` (new repo, original untouched)

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────
# Run once per Colab session
!pip install -q \
    transformers==4.41.2 \
    datasets==2.19.2 \
    accelerate==0.30.0 \
    evaluate==0.4.2 \
    jiwer==3.0.4 \
    sentencepiece==0.2.0 \
    huggingface_hub==0.23.3 \
    tensorboard

!apt-get install -q ffmpeg

print('✅ Dependencies installed')

In [ ]:
# ── CELL 2: Mount Google Drive ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Verify your dataset paths exist
DRIVE_DATASETS = '/content/drive/MyDrive/classroom_project/datasets'
EXPECTED_SPLITS = [
    'processed_kn_1', 'processed_kn_2',
    'processed_hi_1', 'processed_hi_2',
    'processed_en'
]

print('Checking dataset paths...')
for split in EXPECTED_SPLITS:
    path = os.path.join(DRIVE_DATASETS, split)
    exists = os.path.isdir(path)
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {path}')

print('\nDrive mounted and paths verified.')

In [ ]:
# ── CELL 3: Login to HuggingFace (token from your .env) ────────
# Paste your HF_TOKEN here — same token from your .env file
HF_TOKEN    = 'hf_REDACTED_TOKEN'  # from your .env HF_TOKEN
HUB_MODEL_V2 = 'udayakumar8214/whisper-classroom-kn-hi-en-v2'  # NEW repo, original untouched
BASE_MODEL   = 'openai/whisper-medium'   # Upgrade: medium > small for 85%+ accuracy

from huggingface_hub import login
login(token=HF_TOKEN)
print(f'✅ Logged in to HuggingFace')
print(f'   Base model  : {BASE_MODEL}')
print(f'   Will publish: {HUB_MODEL_V2}')

In [ ]:
# ── CELL 4: Load all 5 dataset splits from Drive ───────────────
import gc
from datasets import load_from_disk, concatenate_datasets

DRIVE_DATASETS = '/content/drive/MyDrive/classroom_project/datasets'
SPLIT_NAMES = [
    'processed_kn_1', 'processed_kn_2',
    'processed_hi_1', 'processed_hi_2',
    'processed_en'
]

datasets_list = []
total_rows = 0
for name in SPLIT_NAMES:
    path = os.path.join(DRIVE_DATASETS, name)
    ds = load_from_disk(path)
    print(f'  {name}: {len(ds):,} rows | columns: {ds.column_names}')
    datasets_list.append(ds)
    total_rows += len(ds)

full_dataset = concatenate_datasets(datasets_list).shuffle(seed=42)
del datasets_list; gc.collect()

print(f'\n✅ Total rows: {total_rows:,}')
print(f'   Columns   : {full_dataset.column_names}')

In [ ]:
# ── CELL 5: Validate dataset columns ───────────────────────────
# Your existing splits should already have input_features + labels
# This cell checks and re-processes ONLY if needed

sample = full_dataset[0]
has_features = 'input_features' in sample
has_labels   = 'labels' in sample

print(f'input_features present: {has_features}')
print(f'labels present        : {has_labels}')

if has_features and has_labels:
    print('\n✅ Dataset is already preprocessed for Whisper — skip to Cell 6')
else:
    print('\n⚠️  Dataset needs preprocessing. Running Cell 5b...')

In [ ]:
# ── CELL 5b: Re-preprocess (ONLY run if Cell 5 shows missing columns) ──
# Skip this cell if Cell 5 printed '✅ Dataset is already preprocessed'

from transformers import WhisperProcessor
import numpy as np

processor_check = WhisperProcessor.from_pretrained(BASE_MODEL, language=None, task='transcribe')

# Detect which audio/text column names your dataset uses
sample_cols = full_dataset.column_names
audio_col = 'audio' if 'audio' in sample_cols else None
text_col  = ('sentence' if 'sentence' in sample_cols
              else 'transcription' if 'transcription' in sample_cols
              else 'text' if 'text' in sample_cols else None)

print(f'Audio column : {audio_col}')
print(f'Text column  : {text_col}')

if audio_col and text_col:
    def preprocess(batch):
        audio = batch[audio_col]
        arr = audio['array'] if isinstance(audio, dict) else np.array(audio)
        sr  = audio.get('sampling_rate', 16000) if isinstance(audio, dict) else 16000
        feat = processor_check(
            arr, sampling_rate=sr, return_tensors='pt'
        ).input_features[0]
        batch['input_features'] = feat.numpy() if hasattr(feat, 'numpy') else feat
        batch['labels'] = processor_check.tokenizer(batch[text_col]).input_ids
        return batch

    print('Re-processing dataset...')
    remove_cols = [c for c in sample_cols if c not in ('input_features', 'labels')]
    full_dataset = full_dataset.map(
        preprocess,
        remove_columns=remove_cols,
        num_proc=2,
        desc='Preprocessing'
    )
    print('✅ Re-processing done')
else:
    print('❌ Could not detect audio/text columns. Check column names:', sample_cols)

In [ ]:
# ── CELL 6: Train / Eval split (95 / 5) ───────────────────────
split = full_dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
eval_ds  = split['test']

print(f'Train : {len(train_ds):,} samples')
print(f'Eval  : {len(eval_ds):,} samples')

In [ ]:
# ── CELL 7: Load Whisper-Medium + Processor ────────────────────
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

processor = WhisperProcessor.from_pretrained(
    BASE_MODEL, language=None, task='transcribe'
)

model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)

# Enable multilingual auto-detect (no forced language)
model.generation_config.language           = None
model.generation_config.task               = 'transcribe'
model.generation_config.forced_decoder_ids = None

print(f'✅ whisper-medium loaded ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)')

In [ ]:
# ── CELL 8: Data Collator ──────────────────────────────────────
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class SpeechCollator:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad input features
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        # Pad labels and mask padding with -100
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # Remove decoder start token if prepended
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

data_collator = SpeechCollator(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)
print('✅ Data collator ready')

In [ ]:
# ── CELL 9: WER Metric ─────────────────────────────────────────
import evaluate

wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    print(f'[Eval] WER = {wer:.2f}%  (target < 15%)')
    return {'wer': round(wer, 2)}

print('✅ Metric ready')

In [ ]:
# ── CELL 10: Training Arguments ────────────────────────────────
# Tuned for A100 (40 GB VRAM). Targets WER < 15% (accuracy > 85%)
import transformers
from packaging import version
from transformers import Seq2SeqTrainingArguments

OUTPUT_DIR = '/content/drive/MyDrive/classroom_project/models/whisper-medium-v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Handle old vs new transformers API
use_new_api = version.parse(transformers.__version__) >= version.parse('4.41.0')
eval_key    = 'eval_strategy' if use_new_api else 'evaluation_strategy'

training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,

    # ── Batch size: A100 can handle 16 comfortably ──
    per_device_train_batch_size = 16,
    gradient_accumulation_steps = 1,
    per_device_eval_batch_size  = 8,

    # ── Learning schedule ────────────────────────────
    learning_rate               = 1e-5,
    warmup_steps                = 200,
    max_steps                   = 4000,  # ~4 hrs on A100; increase to 6000 for max accuracy

    # ── Memory / speed ───────────────────────────────
    gradient_checkpointing      = True,
    fp16                        = True,
    dataloader_num_workers      = 4,

    # ── Eval + save every 500 steps ──────────────────
    **{eval_key: 'steps'},
    save_strategy               = 'steps',
    eval_steps                  = 500,
    save_steps                  = 500,
    save_total_limit            = 3,

    # ── Generation ───────────────────────────────────
    predict_with_generate       = True,
    generation_max_length       = 225,

    # ── Logging ──────────────────────────────────────
    logging_steps               = 25,
    report_to                   = ['tensorboard'],

    # ── Best model + Hub push ────────────────────────
    load_best_model_at_end      = True,
    metric_for_best_model       = 'wer',
    greater_is_better           = False,
    push_to_hub                 = True,
    hub_model_id                = HUB_MODEL_V2,
    hub_token                   = HF_TOKEN,
)

print('✅ Training arguments set')
print(f'   Output dir : {OUTPUT_DIR}')
print(f'   Max steps  : {training_args.max_steps}')
print(f'   Batch size : {training_args.per_device_train_batch_size}')
print(f'   FP16       : {training_args.fp16}')

In [ ]:
# ── CELL 11: Build Trainer ─────────────────────────────────────
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args          = training_args,
    model         = model,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
    processing_class = processor,   # works transformers 4.40 → 5.x
)

print('✅ Trainer built and ready')

In [ ]:
# ── CELL 12: START TRAINING ────────────────────────────────────
# ⚠️  This will run for ~3-4 hours on A100.
# Checkpoints are saved to Drive every 500 steps — safe against disconnects.
# Watch the WER column: target is < 15.0%

print('🚀 Starting whisper-medium fine-tuning...')
print(f'   Publishing to : {HUB_MODEL_V2}')
print(f'   Checkpoints   : {OUTPUT_DIR}')
print('─' * 60)

trainer.train()

print('─' * 60)
print('✅ Training complete!')

In [ ]:
# ── CELL 13: Push best model to HuggingFace ───────────────────
# Pushes the best checkpoint (lowest WER) to HF Hub as v2

print(f'Pushing best model to: {HUB_MODEL_V2}')
trainer.push_to_hub()
processor.push_to_hub(HUB_MODEL_V2, token=HF_TOKEN)

print(f'✅ Model published!')
print(f'   🔗 https://huggingface.co/{HUB_MODEL_V2}')

In [ ]:
# ── CELL 14: Per-language WER Evaluation ──────────────────────
# Runs separate WER per language on the eval split
# Requires: langdetect  (pip install langdetect)

!pip install -q langdetect
from langdetect import detect, LangDetectException

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

import numpy as np

results_by_lang = {'kn': {'preds': [], 'refs': []},
                   'hi': {'preds': [], 'refs': []},
                   'en': {'preds': [], 'refs': []},
                   'other': {'preds': [], 'refs': []}}

print('Running per-language evaluation on eval split...')
print('(Processing up to 500 samples for speed)')

eval_sample = eval_ds.select(range(min(500, len(eval_ds))))

for i, sample in enumerate(eval_sample):
    feat = torch.tensor(sample['input_features']).unsqueeze(0).to(device)
    label_ids = [l for l in sample['labels'] if l != -100]
    ref = processor.tokenizer.decode(label_ids, skip_special_tokens=True)

    with torch.inference_mode():
        pred_ids = model.generate(feat)
    pred = processor.tokenizer.decode(pred_ids[0], skip_special_tokens=True)

    # Detect language from reference text
    try:
        lang = detect(ref)
        lang = lang if lang in ('kn', 'hi', 'en') else 'other'
    except LangDetectException:
        lang = 'other'

    results_by_lang[lang]['preds'].append(pred)
    results_by_lang[lang]['refs'].append(ref)

    if (i + 1) % 50 == 0:
        print(f'  [{i+1}/500] processed...')

print('\n' + '='*50)
print('PER-LANGUAGE WER RESULTS')
print('='*50)
for lang, data in results_by_lang.items():
    if not data['refs']:
        continue
    wer = 100 * wer_metric.compute(predictions=data['preds'], references=data['refs'])
    acc = 100 - wer
    status = '✅' if acc >= 85 else '⚠️ '
    label = {'kn': 'Kannada', 'hi': 'Hindi', 'en': 'English', 'other': 'Other'}.get(lang, lang)
    print(f'  {status} {label:10s}: WER={wer:.1f}%  Accuracy≈{acc:.1f}%  (n={len(data["refs"])})')
print('='*50)
print('Target: Accuracy > 85% (WER < 15%) for all languages')

In [ ]:
# ── CELL 15: Update your app to use the new model ─────────────
# After confirming WER < 15%, update your backend .env:
#
#   WHISPER_MODEL_ID=udayakumar8214/whisper-classroom-kn-hi-en-v2
#
# The old model (whisper-classroom-kn-hi-en) is untouched on HuggingFace.
# You can rollback anytime by reverting the .env change.

print('Next steps:')
print(f'  1. Confirm WER < 15% in Cell 14 above')
print(f'  2. Open your .env file on your local machine')
print(f'  3. Change: WHISPER_MODEL_ID={HUB_MODEL_V2}')
print(f'  4. Restart backend: uvicorn main:app --reload')
print(f'  5. Test with a Hindi and Kannada audio file')
print(f'\n  Model URL: https://huggingface.co/{HUB_MODEL_V2}')